# Batch cell typing – IHOPE project

Runs normalization, AnnData construction, GMM thresholding, rule-based cell typing, and summary export for all 13 samples. Starts from pre-filtered CSVs.

In [ ]:
import sys
from pathlib import Path
import gc

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from scripts.transforms import apply_transform
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad
from scripts.annotation import compute_positivity_matrix
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE

## Sample list

`file_basename` is the name used to find the input CSV on disk. `out_basename` is the name used for all outputs. They differ where files have been renamed or used a non-standard suffix.

In [ ]:
# (file_basename, input_suffix, out_basename)
# input_suffix is either "_cleaned_filtered" or "_cleaned_filtered_looped"
SAMPLES = [
    ("IHOPE14_MedLN_BottomLeft",  "_cleaned_filtered",        "IHOPE14_MedLN_BottomLeft"),
    ("IHOPE14_MedLN_TopRight",    "_cleaned_filtered_looped", "IHOPE14_MedLN_TopRight"),
    ("IHOPE14_MedLN_BottomRight", "_cleaned_filtered_looped", "IHOPE14_MedLN_BottomRight"),
    ("IHOPE14_mesLN",             "_cleaned_filtered_looped", "IHOPE14_MesLN"),
    ("IHOPE20_LN",                "_cleaned_filtered",        "IHOPE20_MedLN"),
    ("IHOPE20_Spleen",            "_cleaned_filtered_looped", "IHOPE20_Spleen"),
    ("IHOPE26_LN",                "_cleaned_filtered",        "IHOPE26_MedLN"),
    ("IHOPE26_Spleen",            "_cleaned_filtered",        "IHOPE26_Spleen"),
    ("IHOPE27_LN",                "_cleaned_filtered",        "IHOPE27_MedLN"),
    ("IHOPE27_Spleen",            "_cleaned_filtered",        "IHOPE27_Spleen"),
    ("IHOPE39_LN",                "_cleaned_filtered",        "IHOPE39_MedLN"),
    ("IHOPE39_MesLN_A",           "_cleaned_filtered",        "IHOPE39_MesLN_A"),
    ("IHOPE39_MesLN_B",           "_cleaned_filtered",        "IHOPE39_MesLN_B"),
    ("IHOPE39_Spleen",            "_cleaned_filtered",        "IHOPE39_Spleen"),
]

## Batch processing

For each sample: normalize from the cleaned filtered CSV, build AnnData, run GMM thresholding, assign cell types, and export summary CSV.

In [ ]:
import importlib
from scripts import transforms
importlib.reload(transforms)
from scripts.transforms import apply_transform

In [ ]:
failed = []

for file_basename, input_suffix, out_basename in SAMPLES:
    print(f"Processing: {out_basename}")

    try:
        # Normalize
        input_file  = f"../data/processed/{file_basename}{input_suffix}.csv"
        zscore_file = f"../data/processed/{out_basename}_cleaned_filtered_zscore.csv"

        df_out, markers, metadata, fig = apply_transform(
            input_file=input_file,
            method="zscore",
            output_file=zscore_file,
            save_plot=False,
        )
        print(f"  Normalized: {len(df_out)} cells, {len(markers)} markers")

        # Build AnnData
        adata = load_and_build_anndata(zscore_file)
        del df_out
        gc.collect()
        print(f"  AnnData built: {adata.shape}")

        # GMM thresholding
        adata, thresholds, best_gmms = compute_positivity_matrix(
            adata,
            quantile=0.8,
            random_state=0,
        )
        print(f"  GMM thresholding done")

        # Save AnnData
        h5ad_gmm = f"../data/processed/anndata/{out_basename}_filtered_zscore_GMM.h5ad"
        save_h5ad(adata, h5ad_gmm)
        print(f"  Saved: {h5ad_gmm}")

        # Rule-based cell typing
        adata = assign_cell_types_bool_IHOPE(adata)
        print(f"  Cell types assigned")

        # Save AnnData with cell types
        h5ad_typed = f"../data/processed/anndata/{out_basename}_filtered_zscore_GMM_IHOPE_celltypes.h5ad"
        save_h5ad(adata, h5ad_typed)
        print(f"  Saved: {h5ad_typed}")

        # Summary CSV
        df_summary = summarize_celltypes_IHOPE(
            adata,
            filename=f"{out_basename}_filtered_zscore_IHOPE_summary.csv",
        )
        print(f"  Summary exported")

    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append((out_basename, str(e)))

print(f"\nBatch complete.")
if failed:
    print(f"Failed samples ({len(failed)}):")
    for name, err in failed:
        print(f"  {name}: {err}")
else:
    print("All samples processed successfully.")